In [1]:
!pip -q install -U anthropic datasets pandas tqdm tenacity scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.

In [2]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
print("Anthropic API key loaded securely.")

Enter your Anthropic API key: ··········
Anthropic API key loaded securely.


In [5]:
import os
import json
import time
import re
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from anthropic import (
    Anthropic,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    AuthenticationError
)

from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"

# Try the user's dataset first. If it fails, use the official FiNER-ORD BIO dataset.
PRIMARY_DATASET_NAME = "ChanceFocus/flare-finer-ord"
FALLBACK_DATASET_NAME = "gtfintechlab/finer-ord-bio"

# First test with 20 or 100.
# Then set LIMIT = None for the full test set.
LIMIT = 100

try:
    dataset = load_dataset(PRIMARY_DATASET_NAME)
    DATASET_NAME = PRIMARY_DATASET_NAME
except Exception as e:
    print(f"Could not load {PRIMARY_DATASET_NAME}.")
    print("Using fallback dataset:", FALLBACK_DATASET_NAME)
    print("Original error:", e)
    dataset = load_dataset(FALLBACK_DATASET_NAME)
    DATASET_NAME = FALLBACK_DATASET_NAME

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Dataset:", DATASET_NAME)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC"
]

SYSTEM_PROMPT = """
You are a financial named entity recognition model.

Task:
Given a list of tokens from a financial news text, assign exactly one BIO label to each token.

Allowed labels:
- O
- B-PER, I-PER
- B-ORG, I-ORG
- B-LOC, I-LOC

Definitions:
- PER: person
- ORG: organization
- LOC: location
- O: not a named entity

Rules:
1. Return exactly one label for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use only the allowed labels.
4. Do not add explanations.
5. Return only valid JSON in this exact format:
{"labels": ["O", "B-ORG", "I-ORG"]}
"""

def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )

import numbers

def normalize_label(label):
    label = str(label).strip()

    # Already correct BIO style
    if label in VALID_LABELS:
        return label

    # Convert FiNER style to BIO style
    if label == "PER_B":
        return "B-PER"
    if label == "PER_I":
        return "I-PER"

    if label == "ORG_B":
        return "B-ORG"
    if label == "ORG_I":
        return "I-ORG"

    if label == "LOC_B":
        return "B-LOC"
    if label == "LOC_I":
        return "I-LOC"

    # Manual numeric-string fallback
    manual_map = {
        "0": "O",
        "1": "B-PER",
        "2": "I-PER",
        "3": "B-LOC",
        "4": "I-LOC",
        "5": "B-ORG",
        "6": "I-ORG"
    }

    if label in manual_map:
        return manual_map[label]

    return "O"


def int_label_to_string(label_id, split_data, label_col):
    label_id = int(label_id)
    feature = split_data.features[label_col]

    # Case 1: Sequence(ClassLabel)
    if hasattr(feature, "feature") and hasattr(feature.feature, "int2str"):
        return feature.feature.int2str(label_id)

    # Case 2: ClassLabel
    if hasattr(feature, "int2str"):
        return feature.int2str(label_id)

    # Manual fallback for FiNER-ORD BIO
    manual_map = {
        0: "O",
        1: "PER_B",
        2: "PER_I",
        3: "LOC_B",
        4: "LOC_I",
        5: "ORG_B",
        6: "ORG_I"
    }

    return manual_map.get(label_id, "O")


def convert_labels(raw_labels, split_data, label_col):
    labels = []

    for x in raw_labels:
        if isinstance(x, numbers.Integral):
            label_name = int_label_to_string(x, split_data, label_col)
            labels.append(normalize_label(label_name))
        else:
            labels.append(normalize_label(x))

    return labels

def parse_claude_json(raw_text):
    raw_text = str(raw_text).strip()

    if raw_text == "":
        raise ValueError("Empty model output")

    try:
        return json.loads(raw_text)
    except JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))

        print("Raw model output:")
        print(repr(raw_text))
        raise

def fix_prediction_length(pred_labels, target_len):
    pred_labels = [normalize_label(x) for x in pred_labels]

    if len(pred_labels) < target_len:
        pred_labels = pred_labels + ["O"] * (target_len - len(pred_labels))

    if len(pred_labels) > target_len:
        pred_labels = pred_labels[:target_len]

    return pred_labels

def bio_to_entities(labels):
    entities = []
    start = None
    ent_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                entities.append((start, i, ent_type))
                start = None
                ent_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                entities.append((start, i, ent_type))
            start = i
            ent_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                ent_type = label_type
            elif ent_type != label_type:
                entities.append((start, i, ent_type))
                start = i
                ent_type = label_type

    return entities

def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_entities = Counter(bio_to_entities(gold_labels))
        pred_entities = Counter(bio_to_entities(pred_labels))

        tp += sum((gold_entities & pred_entities).values())
        fp += sum((pred_entities - gold_entities).values())
        fn += sum((gold_entities - pred_entities).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }

@retry(
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5),
    reraise=True
)
def call_claude_finer_ord(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    user_prompt = (
        f"Number of tokens: {len(tokens)}\n\n"
        f"Tokens:\n{indexed_tokens}\n\n"
        f"Return exactly {len(tokens)} BIO labels as JSON."
    )

    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

    except BadRequestError as e:
        print("\nBadRequestError detail:")
        print(e)
        raise

    except AuthenticationError as e:
        print("\nAuthenticationError detail:")
        print(e)
        raise

    raw_output = "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )

    parsed = parse_claude_json(raw_output)

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed

example = split_data[0]

token_col = get_column_name(example, ["tokens", "token", "gold_token"])
label_col = get_column_name(example, ["tags", "tag", "labels", "label", "gold_label"])

print("Token column:", token_col)
print("Label column:", label_col)

sample_size = len(split_data) if LIMIT is None else min(LIMIT, len(split_data))
eval_data = split_data.select(range(sample_size))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = convert_labels(row[label_col], split_data, label_col)

    try:
        result = call_claude_finer_ord(tokens)
        pred_labels = result.get("labels", [])
        pred_labels = fix_prediction_length(pred_labels, len(gold_labels))
        error = ""

    except BadRequestError as e:
        print(f"\nStopping at row {idx} because of BadRequestError.")
        print(e)
        raise

    except AuthenticationError as e:
        print(f"\nStopping at row {idx} because of AuthenticationError.")
        print(e)
        raise

    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_labels": pred_labels,
        "gold_entities": bio_to_entities(gold_labels),
        "pred_entities": bio_to_entities(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    time.sleep(5)

metrics = compute_metrics(gold_all, pred_all)

print("\n===== Claude Opus 4.7 FiNER-ORD Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)

df.to_csv("Claude_Opus_4_7_FiNER_ORD_Evaluation_Results.csv", index=False)

with open("Claude_Opus_4_7_FiNER_ORD_Metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

Could not load ChanceFocus/flare-finer-ord.
Using fallback dataset: gtfintechlab/finer-ord-bio
Original error: Dataset 'ChanceFocus/flare-finer-ord' doesn't exist on the Hub or cannot be accessed.


Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3262
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 402
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1075
    })
})
Dataset: gtfintechlab/finer-ord-bio
Split: test
Columns: ['tokens', 'tags']
Example: {'tokens': ['H', 'ave', 'you', 'ever', 'felt', 'that', 'sudden', ',', 'intense', 'dread', 'that', 'you', '’re', 'about', 'to', 'die', '?'], 'tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
Token column: tokens
Label column: tags


100%|██████████| 100/100 [12:25<00:00,  7.46s/it]


===== Claude Opus 4.7 FiNER-ORD Evaluation =====
Dataset: gtfintechlab/finer-ord-bio
Split: test
Model: claude-opus-4-7
Samples evaluated: 100
Precision / Correctness Rate: 0.6897
Recall: 0.8421
F1 Score: 0.7583
Exact Match Accuracy: 0.7600
Token Accuracy: 0.9775
TP: 80
FP: 36
FN: 15

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,tokens,gold_labels,pred_labels,gold_entities,pred_entities,exact_match,error
0,0,,"[H, ave, you, ever, felt, that, sudden, ,, intense, dread, that, you, ’re, about, to, die, ?]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
1,1,,"[That, in, less, than, seven, minutes, ,, the, knot, of, fear, and, anxiety, unspooling, at, speed, will, swallow, y...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
2,2,,"[And, all, because, you, failed, to, prepare, !]","[O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O]",[],[],True,
3,3,,"[You, idiot, !]","[O, O, O]","[O, O, O]",[],[],True,
4,4,,"[And, now, you, ’re, on, a, train, carriage, in, a, clammy, panic, desperately, searching, for, hope, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",[],[],True,
